In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import pandas as pd
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import ListedColormap, BoundaryNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime, timedelta

#additional
import struct
import xesmf as xe

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RainRadarHourlyMax"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import FigurePlotting_Class

In [ ]:
#Setup
Region = "TRACER"; Case = "WET"; spinup_hours = "0"
Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"


Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
Region = "PRECIP"; Case = "DIURNAL"; spinup_hours = "12"

Region = "Hawaii"; Case = "WET"; spinup_hours = "12"#; spinup_hours="-16"
Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSubsetting import DataSubsetting_Class

#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class, RadarData_PRECIP_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
####################################
#DATA LOADING AND CALCULATION FUNCTIONS

In [ ]:
#Data Loading and Calculation Functions
def SubsetData_Time(data,yearmonthday):
    data_T = data.sel(time=yearmonthday)
    # print(data_T.time) #testing
    return data_T

def GetMean(variableSubset):
    #(1/A) times integral of phi dA 
    #dA is [(R*cos(Lat)dLon)][RdLat] = R^2 cos(Lat)dLatdLon ==> weight is simply cos(Lat)
    weights = np.cos(np.deg2rad(variableSubset.latitude))
    variableMean = variableSubset.weighted(weights).mean(
        dim=("latitude", "longitude"),
        skipna=True
    )
    return variableMean

In [ ]:
# Model Calculation Functions
####################################

#MAIN FUNCTION

def GetHourlyMax_Model(ModelData, rainVars=("rainnc",), subset=True, printout=False):
    """
    Loop over every output time and build hourly-max maps of:
      (1) rain rate (mm/hr) from differences of accumulated rain between outputs
      (2) 1-km AGL reflectivity (dBZ) from the instantaneous refl10cm_1km field
    The max is taken over time only, so each hour keeps a 2D (lat, lon) map.

    Notes:
      - Rain rate over (t_prev, t] is an interval average (e.g. 5-min), so the
        hourly max underestimates the true model time-step peak.
      - Reflectivity is sampled only at output times (not every time step).
      - The first output time is used only as the starting accumulation.
      - n_samples records how many outputs went into each hour (check partial hours).
    """
    hourlyRain, hourlyRefl, hourLabels, nSamples = [], [], [], []
    runRain = runRefl = None
    currentHour = None
    count = 0
    prevAccum = prevTime = None

    def StoreHour():
        # Save the finished hour's maps and its label
        hourlyRain.append(runRain)
        hourlyRefl.append(runRefl)
        hourLabels.append(currentHour)
        nSamples.append(count)

    for t in tqdm(ModelData.timeStrings):
        tDatetime = ParseTime(t)

        # --- load fields at this output time ---
        data = ModelData.GetDataTimestep(t, printout=printout)
        dataDiag = ModelData.GetDataTimestep_diag(t, printout=printout)
        accum = PrepareField(GetAccumRain(data, rainVars), ModelData, subset)
        refl = PrepareField(dataDiag["refl10cm_1km"], ModelData, subset)

        # --- first output time: only store the starting accumulation ---
        if prevAccum is None:
            prevAccum, prevTime = accum, tDatetime
            continue

        # --- new hour reached: store the finished hour, reset running maxes ---
        hourEnd = HourEnding(tDatetime)
        if currentHour is not None and hourEnd != currentHour:
            StoreHour()
            runRain = runRefl = None
            count = 0
        currentHour = hourEnd

        # --- rain rate over (prevTime, t] in mm/hr; clip tiny negatives ---
        dtHours = (tDatetime - prevTime).total_seconds() / 3600.0
        rate = ((accum - prevAccum) / dtHours).clip(min=0)

        # --- update running maxima for this hour ---
        runRain = UpdateRunningMax(runRain, rate)
        runRefl = UpdateRunningMax(runRefl, refl)
        count += 1
        prevAccum, prevTime = accum, tDatetime

    # --- store the last hour ---
    if currentHour is not None:
        StoreHour()

    # --- stack hourly maps into one Dataset: (time, lat, lon) ---
    ds = xr.Dataset({
        "rainrate_hourly_max": xr.concat(hourlyRain, dim="time"),
        "refl1km_hourly_max": xr.concat(hourlyRefl, dim="time"),
    })
    ds = ds.assign_coords(
        time=np.array(hourLabels, dtype="datetime64[ns]"),
        n_samples=("time", np.array(nSamples)),
    )
    ds["rainrate_hourly_max"].attrs.update(
        units="mm/hr",
        description=f"Hourly max of output-interval rain rate from {'+'.join(rainVars)}",
    )
    ds["refl1km_hourly_max"].attrs.update(
        units="dBZ",
        description="Hourly max of refl10cm_1km sampled at output times",
    )
    ds.attrs.update(
        region=ModelData.region, case=ModelData.case,
        spinup_hours=ModelData.spinup_hours, microphysics=ModelData.mpType,
        time_convention="hour-ending, right-closed bins (HH:00, HH+1:00]",
    )
    return ds

#HELPER FUNCTIONS

def ParseTime(t):
    # MPAS time string "YYYY-MM-DD_HH.MM.SS" -> datetime
    return datetime.strptime(t, "%Y-%m-%d_%H.%M.%S")

def HourEnding(tDatetime):
    # Hour bin label = end of the hour (right-closed bins: (HH:00, HH+1:00]).
    # e.g. 00:05...01:00 -> 01:00, matching the hour-ending MRMS QPE convention.
    floor = tDatetime.replace(minute=0, second=0, microsecond=0)
    return floor if tDatetime == floor else floor + timedelta(hours=1)

def GetAccumRain(data, rainVars=("rainnc",)):
    # Total accumulated rain (mm) at one output time.
    # Adds bucket resets (i_rainnc * bucket size) if the counter is in the file,
    # so differences do not go negative when rainnc resets after 100 mm.
    total = 0
    for v in rainVars:
        accum = data[v]
        counter = f"i_{v}"
        if counter in data:
            bucket = float(data.attrs.get(f"config_bucket_{v}", 100.0))
            accum = accum + data[counter] * bucket
        total = total + accum
    return total

def PrepareField(field, ModelData, subset=True):
    # Drop the length-1 time dim, optionally subset to the analysis region,
    # and load into memory so the running max is fast.
    field = field.squeeze(drop=True)
    if subset:
        field = DataSubsetting_Class.SubsetDataRegion(field, ModelData)
    return field.load()

def UpdateRunningMax(runningMax, field):
    # Element-wise max at each grid point (max over TIME only -> still a 2D map).
    # np.fmax ignores NaNs unless both values are NaN.
    if runningMax is None:
        return field.copy()
    return np.fmax(runningMax, field)

def LoadOrRun_GetHourlyMax_Model(ModelData, rainVars=("rainnc",), subset=True, overwrite=False):
    """
    Load hourly-max fields from disk if saved; otherwise compute with
    GetHourlyMax_Model and save to NetCDF. Returns an xarray Dataset.
    """
    os.makedirs(outputDirectory, exist_ok=True)
 
    # --- file name includes the run and the rain variables used ---
    fileName = (
        f"{ModelData.region}_"
        f"{ModelData.case}_"
        f"{ModelData.spinup_hours}hrs_"
        f"{ModelData.mpType}_"
        f"{'+'.join(rainVars)}_"
        f"Model_HourlyMax_RainRate_Refl1km.nc"
    )
    filePath = os.path.join(outputDirectory, fileName)
 
    # --- LOAD ---
    if os.path.exists(filePath) and not overwrite:
        print("Loading hourly-max fields from disk:")
        print(f"  {filePath}")
        with xr.open_dataset(filePath) as ds:
            return ds.load()  # load into memory so the file can close
 
    # --- RUN ---
    print(f"Computing hourly-max fields for {ModelData.mpType}")
    ds = GetHourlyMax_Model(ModelData, rainVars=rainVars, subset=subset)
 
    # --- SAVE ---
    ds.to_netcdf(filePath)
    print("Saved hourly-max fields to disk:")
    print(f"  {filePath}")
    return ds

In [ ]:
#DATA LOADING FOR MRMS QPE DATA

def CorrectSimulationDates(ModelData, simulationDates):
    if int(ModelData.spinup_hours) <= 0:
        # Convert first date to Timestamp
        first_date = pd.to_datetime(simulationDates[0])
        # Subtract one day
        prev_date = (first_date - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        # Prepend
        simulationDates = [prev_date] + simulationDates
    return simulationDates
    
def GetMRMS_QPE_DataDirectory(ModelData, product="MultiSensor_QPE_01H_Pass2_00.00"):
    simulationDates = ModelData.simulationDates
    simulationDates2 = CorrectSimulationDates(ModelData, simulationDates)

    inputDirectory = os.path.join(DirectoryManager.dataDirectory,
             f"Observation_Data/{ModelData.region}/MRMS_RadarData",
             f"{simulationDates2[0]}_{simulationDates2[-1]}",product)
    return inputDirectory
    
def ReadPrecipData_MRMS_QPE(ModelData, yearmonthdayhour, inputDirectory):
    yearmonthday = yearmonthdayhour[0:8]
    hour = yearmonthdayhour[8:]

    MRMS_region = "CONUS" if ModelData.region=="TRACER" else "HAWAII"
    inputPath = os.path.join(inputDirectory,f"MRMSQPE_{MRMS_region}_{ModelData.region}_{yearmonthday}-{hour}0000.nc")
    try:
        precipData = xr.open_dataset(inputPath)["MultiSensor_QPE_01H_Pass2_00.00"].isel(time=0)
    except:
        print(f"{inputPath} does not exist in MRMS data ==> skipping")
        precipData = None
    #Note: data is in mm units
    return precipData

def GetAccumulatedPrecipData_MRMS_QPE(ModelData): 
    #getting inputDirectory
    inputDirectory = GetMRMS_QPE_DataDirectory(ModelData)
    print(f"reading from {inputDirectory}")

    #getting yearmonthdayhour list
    dt_list = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in ModelData.timeStrings]
    start = dt_list[0]
    target = start# + timedelta(hours=13) #only using for rainfall histogram
    # print(f"start_time: {target}")
    
    # Find the entry closest to +12 hours
    closest = min(dt_list, key=lambda x: abs(x - target))
    
    closest_idx = dt_list.index(closest)
    times = ModelData.timeStrings[closest_idx:]
    yearmonthdayhours = sorted({t.replace('-', '').replace('_', '').replace('.', '')[:10] 
                                for t in times})

    precipData_T = None
    precipData_T_Subset_Mean = []; timeList = []
    for count, yearmonthdayhour in tqdm(enumerate(yearmonthdayhours), total=len(yearmonthdayhours)):
        # print(yearmonthdayhour)
        
        precipData = ReadPrecipData_MRMS_QPE(ModelData,yearmonthdayhour,inputDirectory)
        if precipData is None:
            continue #skips missing hours
        if precipData_T is None:
            precipData_T = precipData #equates at first timestep
        else:
            precipData_T += precipData
            
        # --- make a copy for coordinate fixes ---
        precipData_T_2 = precipData_T.assign_coords(
            longitude = precipData_T.longitude - 360
        ).sortby("latitude")

        # --- subset the region ---
        precipData_T_Subset = DataSubsetting_Class.SubsetDataRegion(
            precipData_T_2, ModelData
        )

        # # --- applying radarDataMask --- #decided not to use here
        # RadarDataMask_interp = RadarDataMask.astype(float).interp(latitude = precipData_T_Subset.latitude,
        #                                         longitude = precipData_T_Subset.longitude,
        #                                         method='nearest')
        # RadarDataMask_interp = RadarDataMask_interp ==1
        # precipData_T_Subset=precipData_T_Subset.where(RadarDataMask_interp == True)

        # --- mean of the subset domain ---
        meanValue = GetMean(precipData_T_Subset).item()
        precipData_T_Subset_Mean.append(meanValue)
        timeList.append(yearmonthdayhour)

    timeList = [datetime.strptime(t, "%Y%m%d%H") for t in timeList]
    return np.array(precipData_T_Subset_Mean),np.array(timeList)

def LoadOrRun_GetAccumulatedPrecipData_MRMS_QPE(
    ModelData,
    overwrite=False
):
    """
    Load or compute MRMS QPE domain-mean accumulated precipitation time series.
    Always returns Python datetime.datetime objects for time.
    """

    os.makedirs(outputDirectory, exist_ok=True)

    fileName = (
        f"{ModelData.region}_"
        f"{ModelData.case}_"
        f"{ModelData.spinup_hours}hrs_"
        f"MRMS_QPE_Precipitation_Timeseries.nc"
    )
    filePath = os.path.join(outputDirectory, fileName)

    # --------------------------------------------------
    # LOAD
    # --------------------------------------------------
    if os.path.exists(filePath) and not overwrite:
        print("Loading MRMS QPE precip time series from disk:")
        print(f"  {filePath}")

        with xr.open_dataset(filePath) as ds:
            precipMean = ds["precip_mean"].values
            time_np = ds["time"].values

        # 🔑 Convert numpy.datetime64 → datetime.datetime
        timeArray = np.array(
            [t.astype("datetime64[ms]").astype(datetime) for t in time_np],
            dtype=object
        )

        return precipMean, timeArray

    # --------------------------------------------------
    # RUN
    # --------------------------------------------------
    print("Computing MRMS QPE accumulated precip time series")
    precipMean, timeArray = GetAccumulatedPrecipData_MRMS_QPE(ModelData)
    # timeArray is already Python datetime here

    # --------------------------------------------------
    # SAVE
    # --------------------------------------------------
    ds = xr.Dataset(
        data_vars={
            "precip_mean": ("time", precipMean),
        },
        coords={
            # NetCDF-safe datetime
            "time": np.array(timeArray, dtype="datetime64[ms]"),
        }
    )

    ds["precip_mean"].attrs["description"] = (
        "Domain-mean accumulated MRMS QPE precipitation"
    )
    ds.attrs["dataset"] = "MRMS_QPE"
    ds.attrs["region"] = ModelData.region
    ds.attrs["case"] = ModelData.case
    ds.attrs["spinup_hours"] = ModelData.spinup_hours

    ds.to_netcdf(filePath)
    print("Saved MRMS QPE precip time series to disk:")
    print(f"  {filePath}")

    return precipMean, timeArray


In [ ]:
#Observation Calculation Functions

#MAIN FUNCTION

def GetHourlyMax_Obs(obsTimes, LoadField, startTime, endTime, varName="refl1km_hourly_max"):
    """
    Running hourly max (per grid point, over time only) for an observed 2D field.
      obsTimes  : list of datetimes with available observations (e.g. MRMS every 2 min)
      LoadField : function t -> 2D DataArray (latitude, longitude), or None if missing
      startTime, endTime : model period; only obs in (startTime, endTime] are used
    Returns a Dataset with varName(time, latitude, longitude) and n_samples(time).
    """
    hourlyMaps, hourLabels, nSamples = [], [], []
    runMax = None
    currentHour = None
    count = 0

    def StoreHour():
        # Save the finished hour's map and its label
        hourlyMaps.append(runMax)
        hourLabels.append(currentHour)
        nSamples.append(count)

    # --- keep obs inside the model period (first model time is not binned) ---
    times = sorted(t for t in obsTimes if startTime < t <= endTime)

    for t in tqdm(times):
        field = LoadField(t)
        if field is None:
            continue  # missing file -> skipped (shows up as lower n_samples)

        # --- new hour reached: store the finished hour, reset running max ---
        hourEnd = HourEnding(t)
        if currentHour is not None and hourEnd != currentHour:
            StoreHour()
            runMax = None
            count = 0
        currentHour = hourEnd

        # --- update running max for this hour ---
        runMax = UpdateRunningMax(runMax, field)
        count += 1

    # --- store the last hour ---
    if currentHour is not None:
        StoreHour()

    # --- stack hourly maps into one Dataset: (time, lat, lon) ---
    ds = xr.Dataset({varName: xr.concat(hourlyMaps, dim="time")})
    ds = ds.assign_coords(
        time=np.array(hourLabels, dtype="datetime64[ns]"),
        n_samples=("time", np.array(nSamples)),
    )
    ds.attrs["time_convention"] = "hour-ending, right-closed bins (HH:00, HH+1:00]"
    return ds


#HELPER FUNCTIONS

def Flatten2DLatLon(da, latName="latitude", lonName="longitude"):
    """If lat/lon are 2D on a regular grid, convert them to 1D latitude/longitude dimension coords."""
    lat, lon = da[latName], da[lonName]
    if lat.ndim == 1 and lon.ndim == 1:
        return da                                            # already 1D: nothing to do
    if lat.ndim != 2 or lon.ndim != 2 or lat.dims != lon.dims:
        raise ValueError(f"Unexpected lat/lon shapes: {lat.dims}, {lon.dims}")
    d0, d1 = lat.dims
    latV, lonV = lat.values, lon.values
    if np.allclose(latV, latV[:, [0]]) and np.allclose(lonV, lonV[[0], :]):
        latDim, lonDim, lat1, lon1 = d0, d1, latV[:, 0], lonV[0, :]     # lat varies along dim 0
    elif np.allclose(latV, latV[[0], :]) and np.allclose(lonV, lonV[:, [0]]):
        latDim, lonDim, lat1, lon1 = d1, d0, latV[0, :], lonV[:, 0]     # lat varies along dim 1
    else:
        raise ValueError("lat/lon are curvilinear (not a regular lat-lon grid); needs regridding")
    da = da.drop_vars([latName, lonName]).rename({latDim: latName, lonDim: lonName})
    return da.assign_coords({latName: lat1, lonName: lon1})

def FixObsGrid(field, ModelData):
    field = Flatten2DLatLon(field)          # NEW: 2D lat/lon -> 1D (no change if already 1D)
    if float(field.longitude.max()) > 180:
        field = field.assign_coords(longitude=field.longitude - 360)
    field = field.sortby("latitude")
    return DataSubsetting_Class.SubsetDataRegion(field, ModelData)

def CleanObsRefl(refl, missingBelow=-90.0, floor=-30.0):
    # Missing/no-coverage flags -> NaN; weak/no-echo values -> floor value,
    # so no-echo areas do not spread NaNs during vertical interpolation.
    # CHECK the fill values in your MRMS/S-Pol files and adjust missingBelow.
    refl = refl.where(refl > missingBelow)
    return refl.where(refl.isnull() | (refl > floor), floor)

def GetTerrainOnGrid(ModelData, grid):
    # Model terrain height (m MSL) from the static data, interpolated to the obs lat/lon grid.
    # Using model terrain keeps obs sampling consistent with the model's AGL definition.
    ter = ModelData.GetStaticData(ModelData.dataDirectory)[0].ter
    if "time" in ter.dims:                     # drop a time dim if present
        ter = ter.isel(time=0)
    ter = ter.squeeze(drop=True)
    return ter.interp(latitude=grid.latitude, longitude=grid.longitude)

def InterpTo1kmAGL(refl3D, terrain, heightDim="height", heightScale=1.0, aglHeight=1000.0):
    # Sample each column at z_MSL = 1 km + terrain (linear interpolation in height).
    # heightScale converts the obs height coordinate to meters (1000.0 if it is in km).
    # Points below the lowest or above the highest level return NaN.
    refl3D = refl3D.transpose(heightDim, ...)
    z = refl3D[heightDim].values * heightScale
    vals = refl3D.values
    target = (aglHeight + terrain).values

    # --- bracketing levels for each column ---
    idx = np.clip(np.searchsorted(z, target), 1, len(z) - 1)
    z0, z1 = z[idx - 1], z[idx]
    v0 = np.take_along_axis(vals, (idx - 1)[None], axis=0)[0]
    v1 = np.take_along_axis(vals, idx[None], axis=0)[0]

    # --- linear interpolation; NaN outside the level range ---
    w = (target - z0) / (z1 - z0)
    out = v0 + w * (v1 - v0)
    out = np.where((target < z[0]) | (target > z[-1]) | np.isnan(target), np.nan, out)
    return xr.DataArray(out, coords=terrain.coords, dims=terrain.dims)

def MakeLoader_Refl1kmAGL(LoadObsRefl3D, ModelData, heightDim="height", heightScale=1.0):
    """
    Wrap your existing obs reader into a function t -> 1-km AGL reflectivity.
      LoadObsRefl3D : your reader, t -> 3D DataArray (height, latitude, longitude) or None
    Terrain is interpolated once, on the first loaded grid, and reused.
    """
    cache = {}

    def LoadField(t):
        refl3D = LoadObsRefl3D(t)
        if refl3D is None:
            return None
        refl3D = CleanObsRefl(FixObsGrid(refl3D, ModelData))
        if "terrain" not in cache:
            cache["terrain"] = GetTerrainOnGrid(ModelData, refl3D)
        return InterpTo1kmAGL(refl3D, cache["terrain"], heightDim, heightScale).load()

    return LoadField


#LOAD OR RUN

def LoadOrRun_GetHourlyMax_Obs(ModelData, obsTimes, LoadField, obsName="MRMS",
                               varName="refl1km_hourly_max", overwrite=False):
    """
    Load observed hourly-max fields from disk if saved; otherwise compute and save.
    """
    os.makedirs(outputDirectory, exist_ok=True)
    fileName = (
        f"{ModelData.region}_"
        f"{ModelData.case}_"
        f"{ModelData.spinup_hours}hrs_"
        f"{obsName}_{varName}.nc"
    )
    filePath = os.path.join(outputDirectory, fileName)

    # --- LOAD ---
    if os.path.exists(filePath) and not overwrite:
        print(f"Loading {obsName} hourly-max fields from disk:\n  {filePath}")
        with xr.open_dataset(filePath) as ds:
            return ds.load()

    # --- RUN: same period as the model run ---
    startTime = ParseTime(ModelData.timeStrings[0])
    endTime = ParseTime(ModelData.timeStrings[-1])
    print(f"Computing {obsName} hourly-max fields")
    ds = GetHourlyMax_Obs(obsTimes, LoadField, startTime, endTime, varName=varName)
    ds.attrs.update(region=ModelData.region, case=ModelData.case,
                    spinup_hours=ModelData.spinup_hours, dataset=obsName)

    # --- SAVE ---
    ds.to_netcdf(filePath)
    print(f"Saved {obsName} hourly-max fields to disk:\n  {filePath}")
    return ds

In [ ]:
# Data Loaders
#--- Rainfall ---
def LoadMRMSQPE(t):
    # QPE file at HH:00 is the accumulation over the previous hour (mm = mm/hr)
    inputDirectory = GetMRMS_QPE_DataDirectory(ModelData_NSSL)
    precip = ReadPrecipData_MRMS_QPE(ModelData_NSSL, t.strftime("%Y%m%d%H"), inputDirectory)
    if precip is None:
        return None
    precip = precip.drop_vars("time", errors="ignore")   # drop scalar time coord
    return FixObsGrid(precip, ModelData_NSSL).load()      # lon fix + region subset

#---Radar Reflectivity---
# def MakeLoader_MRMSRefl1km(RadarData_MRMS, ModelData, maxOffsetMinutes=1):
#     """
#     Returns a function t -> MRMS 1-km MSL reflectivity map (lat, lon), or None.
#     LoadClosestMRMSFile always returns the nearest file, so we skip files already
#     used (duplicates) and files too far from the requested time (gaps).
#     """
#     seenFiles = set()

#     def LoadField(t):
#         radarData, filePath = RadarData_MRMS.LoadClosestMRMSFile(target_time=t)

#         # --- skip duplicates and files too far from t ---
#         if filePath in seenFiles:
#             return None
#         fileTime = pd.to_datetime(radarData["time"].values[0]).to_pydatetime()
#         if abs(fileTime - t) > timedelta(minutes=maxOffsetMinutes):
#             return None
#         seenFiles.add(filePath)

#         # --- 2D map: drop time, fix lon/lat, subset to region ---
#         if isinstance(radarData, xr.Dataset):        # take the reflectivity variable
#             radarData = radarData[list(radarData.data_vars)[0]]
#         refl = radarData.isel(time=0).drop_vars("time", errors="ignore").squeeze(drop=True)
#         return FixObsGrid(refl, ModelData).load()

#     return LoadField

def MakeLoader_MRMSRefl1kmAGL(ModelData, DirectoryManager, RadarObservationLevels,
                              heightDim="heightAboveSea", heightScale=1.0):
    """
    Returns a function t (datetime) -> MRMS reflectivity at 1 km AGL (lat, lon), or None.
    Loads all MRMS levels at the model output time, then samples each column at
    1 km + model terrain (MSL -> AGL).
    """
    # --- model output datetime -> time index (GetData_AllZLevels takes an index) ---
    timeIndex = {ParseTime(s): i for i, s in enumerate(ModelData.timeStrings)}
    cache = {}

    def LoadField(t):
        refl3D = RadarData_MRMS_Class.GetData_AllZLevels(
            DirectoryManager, ModelData, timeIndex[t], RadarObservationLevels)
        if refl3D is None:
            return None

        # --- drop any time dim, fix lon/lat, subset to region, clean fill values ---
        if "time" in refl3D.dims:
            refl3D = refl3D.isel(time=0)
        refl3D = CleanObsRefl(FixObsGrid(refl3D, ModelData))

        # --- model terrain on the obs grid (computed once, then reused) ---
        if "terrain" not in cache:
            cache["terrain"] = GetTerrainOnGrid(ModelData, refl3D)

        return InterpTo1kmAGL(refl3D, cache["terrain"],
                              heightDim=heightDim, heightScale=heightScale).load()

    return LoadField
    
def InterpTo1kmAGL(refl3D, terrain, heightDim="height", heightScale=1.0, aglHeight=1000.0):
    # Sample each column at z_MSL = 1 km + terrain (linear interpolation in height).
    # heightScale converts the obs height coordinate to meters (1000.0 if it is in km).
    # Points below the lowest or above the highest level return NaN.
    refl3D = refl3D.transpose(heightDim, ...)
    z = refl3D[heightDim].values * heightScale
    vals = refl3D.values
    target = (aglHeight + terrain).values

    # --- bracketing levels for each column ---
    idx = np.clip(np.searchsorted(z, target), 1, len(z) - 1)
    z0, z1 = z[idx - 1], z[idx]
    v0 = np.take_along_axis(vals, (idx - 1)[None], axis=0)[0]
    v1 = np.take_along_axis(vals, idx[None], axis=0)[0]

    # --- linear interpolation; NaN outside the level range ---
    w = (target - z0) / (z1 - z0)
    out = v0 + w * (v1 - v0)
    out = np.where((target < z[0]) | (target > z[-1]) | np.isnan(target), np.nan, out)
    return xr.DataArray(out, coords=terrain.coords, dims=terrain.dims)

def CleanObsRefl(refl, missingBelow=-90.0, floor=-30.0):
    # Missing/no-coverage flags -> NaN; weak/no-echo values -> floor value,
    # so no-echo areas do not spread NaNs during vertical interpolation.
    # CHECK the fill values in your MRMS/S-Pol files and adjust missingBelow.
    refl = refl.where(refl > missingBelow)
    return refl.where(refl.isnull() | (refl > floor), floor)

########################################################################################################################
# --- MRMS reflectivity loader (from RadarComparison_MRMS.py) ---
if Region in ['TRACER','Hawaii']:
    if int(ModelData_NSSL.spinup_hours) <= 0:
        if ModelData_NSSL.region == "TRACER":
            if ModelData_NSSL.case == "WET":
                dateString = '2022-06-30_2022-07-03'
            elif ModelData_NSSL.case == "DIURNAL":
                dateString = '2022-06-21_2022-06-24'
        elif ModelData_NSSL.region == "Hawaii":
            if ModelData_NSSL.case == "WET":
                dateString = "2021-12-05_2021-12-08"
            elif ModelData_NSSL.case == "TRADES":
                dateString = "2022-08-07_2022-08-10"
    else:
        dateString = f"{ModelData_NSSL.simulationDates[0]}_{ModelData_NSSL.simulationDates[-1]}"
    
    RadarData_MRMS = RadarData_MRMS_Class(
        ModelData_NSSL,
        fileDirectory=os.path.join(DirectoryManager.dataDirectory,
                                   f"Observation_Data/{ModelData_NSSL.region}/MRMS_RadarData",
                                   dateString,
                                   "MergedReflectivityQC_01.00"))
########################################################################################################################

#--- PRECIP rainfall (QPESUMS) ---
def MakeLoader_PRECIP_QPE(ModelData):
    # Returns t (datetime, hour-ending) -> hourly QPESUMS accumulation map (mm = mm/hr), or None.
    inputDirectory = GetPRECIP_QPE_DataDirectory(ModelData)

    def LoadField(t):
        try:
            precip, nx, ny = ReadPrecipData_PRECIP_QPE(ModelData, t.strftime("%Y%m%d%H"), inputDirectory)
        except (FileNotFoundError, struct.error):
            print(f"QPESUMS file missing for {t} ==> skipping")
            return None
        # --- attach the 12.5-km/1000 degree QPESUMS lat/lon axes ---
        qpeLon, qpeLat = BuildQPELonLat(nx, ny)
        da = xr.DataArray(precip, dims=("latitude", "longitude"),
                          coords={"longitude": qpeLon[0, :], "latitude": qpeLat[:, 0]},
                          name="PRECIP_QPE")
        return FixObsGrid(da, ModelData).load()      # sort latitude + subset region
    return LoadField


def GetPRECIP_QPE_DataDirectory(ModelData):
    simulationDates = ModelData.simulationDates
    simulationDates2 = CorrectSimulationDates(ModelData, simulationDates)

    inputDirectory = os.path.join(DirectoryManager.dataDirectory,
             f"Observation_Data/PRECIP/QPE1HR", ModelData.case)
    return inputDirectory

import struct

def BuildQPELonLat(nx, ny):
    dx = 12.5 / 1000.0
    lon0 = 118.0
    lat0 = 27.0

    lon = lon0 + dx * np.arange(nx)
    lat = lat0 - dx * np.arange(ny)

    qpeLon, qpeLat = np.meshgrid(lon, lat)
    return qpeLon, qpeLat


def ReadQPEData(filePath):
    f=open(filePath,'rb')
    header=f.read(170) # header
    rain=np.reshape(struct.unpack('@'+str(441*561)+'h',f.read(441*561*2)),[441,561],'F')
    rain = rain / 4 #converting from units of 0.25 mm to mm
    rain[rain<0] = np.nan
    rain = rain.T
    rain = rain[::-1]

    ny,nx=rain.shape
    return rain, nx,ny 


def ReadPrecipData_PRECIP_QPE(ModelData,yearmonthdayhour,inputDirectory):
    yearmonthday = yearmonthdayhour[0:8]
    hour = yearmonthdayhour[8:]
    inputFolder = os.path.join(inputDirectory,yearmonthday)
    filePath = os.path.join(inputFolder, f"CB_GC_PCP_1H_RAD.{yearmonthday}.{hour}00")#.gz")
    # print(filePath) #* testing

    precipData,qpeLon,qpeLat = ReadQPEData(filePath)
    return precipData, qpeLon,qpeLat


#--- PRECIP radar reflectivity (S-Pol) at 1 km AGL ---
def MakeLoader_PRECIPRefl1kmAGL(RadarData_PRECIP, ModelData, DirectoryManager,
                                heightDim=None, heightScale=1000.0):
    """
    Returns t (datetime at a model output time) -> S-Pol reflectivity at 1 km AGL, or None.
    Same MSL -> AGL method as MRMS: sample each column at 1 km + model terrain.
    heightScale: 1000.0 if the S-Pol height coordinate is in km.
    """
    timeIndex = {ParseTime(s): i for i, s in enumerate(ModelData.timeStrings)}
    cache = {}

    def LoadField(t):
        # --- same two calls as PRECIP_CFAD.py (GetData_AllZLevels takes a time index) ---
        refl3D, _ = RadarData_PRECIP.GetData_AllZLevels(DirectoryManager, ModelData, timeIndex[t])
        if refl3D is None:
            return None
        refl3D = RadarData_PRECIP.InterpolateRadarData3D(refl3D, ModelData, DirectoryManager)

        # --- reduce to a clean 3D (height, lat, lon) array ---
        if isinstance(refl3D, xr.Dataset):
            refl3D = refl3D[list(refl3D.data_vars)[0]]
        if "time" in refl3D.dims:
            refl3D = refl3D.isel(time=0)
        refl3D = CleanObsRefl(FixObsGrid(refl3D, ModelData))

        # --- find the height dimension; attach z_heights if it has no coordinate ---
        hDim = heightDim or [d for d in refl3D.dims if d not in ("latitude", "longitude")][0]
        if hDim not in refl3D.coords:
            refl3D = refl3D.assign_coords({hDim: np.asarray(RadarData_PRECIP.z_heights)})

        # --- model terrain on this grid (computed once, then reused) ---
        if "terrain" not in cache:
            cache["terrain"] = GetTerrainOnGrid(ModelData, refl3D)

        return InterpTo1kmAGL(refl3D, cache["terrain"],
                              heightDim=hDim, heightScale=heightScale).load()
    return LoadField

In [ ]:
#Code for Calculating Stratiform/Convective Partition Fraction

def LoadOrRun_GetStratiformTimeseries(ModelData, ds_NSSL, ds_TEMPO,
                                      ds_obsRefl=None, ds_obsRain=None,
                                      overwrite=False):
    """
    Load the stratiform-fraction time series from disk if saved; otherwise compute
    with GetStratiformTimeseries (default thresholds and Z-R method) and save.
    Returns a DataFrame indexed by (time, label).
    """
    os.makedirs(outputDirectory, exist_ok=True)

    # --- file name from the run ---
    fileName = (
        f"{ModelData.region}_"
        f"{ModelData.case}_"
        f"{ModelData.spinup_hours}hrs_"
        f"Stratiform_Fraction_Timeseries.pkl"
    )
    filePath = os.path.join(outputDirectory, fileName)

    # --- LOAD ---
    if os.path.exists(filePath) and not overwrite:
        print("Loading stratiform time series from disk:")
        print(f"  {filePath}")
        return pd.read_pickle(filePath)

    # --- RUN ---
    print("Computing stratiform time series")
    series = GetStratiformTimeseries(ds_NSSL, ds_TEMPO,
                                     ds_obsRefl=ds_obsRefl, ds_obsRain=ds_obsRain)

    # --- SAVE ---
    series.to_pickle(filePath)
    print("Saved stratiform time series to disk:")
    print(f"  {filePath}")
    return series

def GetStratiformTimeseries(ds_NSSL, ds_TEMPO, ds_obsRefl=None, ds_obsRain=None,
                            dBZ_threshold=np.array([15, 40])):
    """
    Runs CompareStratiformFractions at every hour in ds_obsRefl.time (or ds_NSSL.time
    if there is no reflectivity obs). A dataset that lacks an exact match for an hour
    is passed as None, so that hour is skipped for it instead of using a nearby hour.
    Returns a DataFrame indexed by (time, label), e.g. (2022-07-01 18:00, "NSSL refl").
    """
    times = (ds_obsRefl if ds_obsRefl is not None else ds_NSSL).time.values

    def HasTime(ds, t):
        # exact hour match only (searchsorted would silently use the next hour)
        return ds if (ds is not None and t in ds.time.values) else None

    frames = {}
    for t in tqdm(times):
        frames[pd.to_datetime(t)] = CompareStratiformFractions(
            HasTime(ds_NSSL, t), HasTime(ds_TEMPO, t),
            ds_obsRefl=HasTime(ds_obsRefl, t), ds_obsRain=HasTime(ds_obsRain, t),
            timeUTC=t, dBZ_threshold=dBZ_threshold)

    return pd.concat(frames, names=["time", "label"]).astype(float)

def CompareStratiformFractions(ds_NSSL, ds_TEMPO, ds_obsRefl=None, ds_obsRain=None,
                               timeUTC="2022-07-01T18:00",
                               dBZ_threshold=np.array([15, 40]),
                               reflVar="refl1km_hourly_max",
                               rainVar="rainrate_hourly_max"):
    """
    Stratiform fractions at one hour for NSSL, TEMPO, and observations,
    from reflectivity (dBZ bounds) and rain rate (Marshall-Palmer bounds).
    Missing observation datasets (None) are skipped.
    Returns a DataFrame with one row per dataset/field.
    """
    # --- rain-rate bounds from the dBZ bounds (Marshall-Palmer) ---
    [R_threshold] = ConvertDBZRainRate(values=dBZ_threshold, source="dbz")

    # --- (label, dataset, variable, bounds) for every comparison ---
    cases = [
        ("NSSL refl",  ds_NSSL,    reflVar, dBZ_threshold),
        ("MRMS refl",  ds_obsRefl, reflVar, dBZ_threshold),
        ("TEMPO refl", ds_TEMPO,   reflVar, dBZ_threshold),
        ("NSSL rain",  ds_NSSL,    rainVar, R_threshold),
        ("MRMS rain",  ds_obsRain, rainVar, R_threshold),
        ("TEMPO rain", ds_TEMPO,   rainVar, R_threshold),
    ]

    # --- compute each fraction, skipping missing datasets ---
    results = {}
    for label, ds, var, bounds in cases:
        if ds is None or var not in ds:
            continue
        out = GetStratiformFraction(ds, var, bounds, timeUTC)
        results[label] = {k: v for k, v in out.items() if k != "time"}
    return pd.DataFrame(results).T[["strat_frac_of_precip", "strat_frac_of_domain"]]

def GetStratiformFraction(ds, var, bounds, timeUTC="2022-07-01T18:00"):
    """
    Stratiform/convective area fractions at one hour from a hourly-max dataset.
      ds     : hourlyMax_NSSL, hourlyMax_TEMPO, hourlyMax_obsRain, or hourlyMax_obsRefl
      var    : "refl1km_hourly_max" (dBZ) or "rainrate_hourly_max" (mm/hr)
      bounds : (lo, hi) in the units of var
      Stratiform: lo <= x < hi.  Convective: x >= hi.  Below lo is excluded from precip.
    Fractions are cos(lat)-weighted; NaN (no coverage) is excluded from every denominator.
    """
    # --- first hour-ending time at or after timeUTC (same rule as PlotHourlyMax) ---
    idx = int(np.searchsorted(ds.time.values, np.datetime64(timeUTC), side="left"))
    field = ds[var].isel(time=idx)

    # --- classes and area weights ---
    lo, hi = bounds
    w = np.cos(np.deg2rad(field.latitude)).broadcast_like(field)
    valid = field.notnull()                    # inside coverage
    strat = (field >= lo) & (field < hi)       # NaN compares as False
    conv = field >= hi

    wStrat = float(w.where(strat).sum())
    wConv = float(w.where(conv).sum())
    wValid = float(w.where(valid).sum())
    wPrecip = wStrat + wConv

    return {
        "time": pd.to_datetime(ds.time.values[idx]),
        "strat_frac_of_precip": wStrat*1e2 / wPrecip if wPrecip > 0 else np.nan,
        "strat_frac_of_domain": wStrat*1e2 / wValid if wValid > 0 else np.nan,
        "conv_frac_of_domain": wConv*1e2 / wValid if wValid > 0 else np.nan,
        "precip_frac_of_domain": wPrecip*1e2 / wValid if wValid > 0 else np.nan,
    }

def ConvertDBZRainRate(values=np.arange(15,60+5,5), source="dbz",
                       method="WSR-88D"):
    """
    Marshall-Palmer Z = a R^b conversion.
      source="dbz"       : input is reflectivity (dBZ)    -> returns rain rate (mm/hr)
      source="rain_rate" : input is rain rate (mm/hr)     -> returns reflectivity (dBZ)
    Works on scalars, numpy arrays, and xarray DataArrays.
    """

    methodsDictionary={"Marshall-Palmer": (200,1.6),
                       "WSR-88D": (300, 1.4)}
    [a, b] = methodsDictionary[method]
            
    if source == "dbz":
        Z = 10.0 ** (values / 10.0)            # dBZ -> linear Z (mm^6 m^-3)
        R = (Z / a) ** (1.0 / b)            # Z -> R (mm/hr)
        return [R]
    elif source == "rain_rate":
        dBZ = 10.0 * np.log10(a * values ** b)   # R -> Z -> dBZ
        return [dBZ]
    else:
        raise ValueError("source must be 'dbz' or 'rain_rate'")

In [ ]:
# Observation Calculation Functions
def GetObservations_Precipitation(ModelData, overwrite=False):
    """
    Hourly-max observed rain rate for this run, from the region's rain product:
      PRECIP        -> QPESUMS hourly QPE
      TRACER/Hawaii -> MRMS hourly QPE
    Returns an xarray Dataset (time, latitude, longitude). Loads from disk if saved.
    """
    # --- hour-ending obs times covering the model period ---
    startTime = ParseTime(ModelData.timeStrings[0])
    endTime = ParseTime(ModelData.timeStrings[-1])
    obsTimes = list(pd.date_range(startTime, endTime, freq="1h").to_pydatetime())

    # --- pick the loader and file tag for this region ---
    if ModelData.region == "PRECIP":
        LoadObsRain = MakeLoader_PRECIP_QPE(ModelData)
        obsRainName = "PRECIP_QPESUMS"
    else:
        LoadObsRain = LoadMRMSQPE
        obsRainName = "MRMS_QPE"

    return LoadOrRun_GetHourlyMax_Obs(
        ModelData, obsTimes, LoadObsRain,
        obsName=obsRainName, varName="rainrate_hourly_max",
        overwrite=overwrite)

def GetObservations_Reflectivity(ModelData, DirectoryManager, overwrite=False):
    """
    Hourly-max observed 1-km AGL reflectivity for this run, from the region's radar:
      PRECIP        -> S-Pol
      TRACER/Hawaii -> MRMS
    Returns an xarray Dataset (time, latitude, longitude). Loads from disk if saved.
    """
    # --- obs are sampled at the model output times ---
    obsTimes = [ParseTime(s) for s in ModelData.timeStrings]

    # --- pick the loader and file tag for this region ---
    if ModelData.region == "PRECIP":
        # S-Pol (PRECIP campaign)
        RadarData_PRECIP = RadarData_PRECIP_Class(
            ModelData,
            os.path.join(DirectoryManager.dataDirectory,
                         "Observation_Data/PRECIP/Radar", ModelData.case))
        LoadObsRefl = MakeLoader_PRECIPRefl1kmAGL(
            RadarData_PRECIP, ModelData, DirectoryManager,
            heightScale=1000.0)                        # 1.0 if heights are in meters
        obsReflName = "PRECIP_SPol_AGL"
    else:
        # MRMS (TRACER, Hawaii)
        RadarObservationLevels = RadarObservationMask_Class.LoadRadarObservationLevels_MRMS(
            DirectoryManager, ModelData)
        LoadObsRefl = MakeLoader_MRMSRefl1kmAGL(
            ModelData, DirectoryManager, RadarObservationLevels,
            heightDim="heightAboveSea", heightScale=1.0)   # 1000.0 if in km
        obsReflName = "MRMS_AGL"

    return LoadOrRun_GetHourlyMax_Obs(
        ModelData, obsTimes, LoadObsRefl,
        obsName=obsReflName, varName="refl1km_hourly_max",
        overwrite=overwrite)

In [ ]:
###################
#PLOTTING FUNCTIONS

In [ ]:
#Single Time Max Field Plot
def PlotHourlyMax(ds_NSSL, ds_TEMPO, ds_obsRain=None, ds_obsRefl=None,
                  timeUTC="2022-07-01T18:00", obsName="MRMS",
                  rainMin=0.3,rainMax=None, reflMin=0,reflMax=None):
    """
    2x3 maps at one hour: rows = rain rate, 1-km reflectivity;
    cols = NSSL, observations, TEMPO. Missing obs -> blank panel.
    Values at or below rainMin / reflMin are masked so no-rain areas stay blank.
    Reflectivity uses the same colormap, levels, ticks, and map features as the
    reflectivity comparison figures (RadarPlotting_Class).
    """
    # --- first hour-ending time at or after timeUTC (from the NSSL run) ---
    timeIndex = int(np.searchsorted(ds_NSSL.time.values, np.datetime64(timeUTC), side="left"))
    time = ds_NSSL.time.isel(time=timeIndex).values

    # --- variables, labels, color limits, and obs dataset for each row ---
    rows = [
        ("rainrate_hourly_max", "Hourly max rain rate (mm/hr)", rainMin, rainMax, "turbo",
         ds_obsRain, f"{obsName} QPE (hourly mean)"),
        ("refl1km_hourly_max", "Hourly max 1-km reflectivity (dBZ)", reflMin, reflMax, None,
         ds_obsRefl, obsName),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(17, 10),
                             subplot_kw={"projection": ccrs.PlateCarree()},
                             gridspec_kw={"wspace": 0.05, "hspace": 0.15,
                                          "left": 0.05, "right": 0.90,
                                          "top": 0.90, "bottom": 0.05})

    for r, (var, label, vmin, vmax, cmap, ds_obs, obsTitle) in enumerate(rows):
        panels = [(ds_NSSL, "NSSL"), (ds_obs, obsTitle), (ds_TEMPO, "TEMPO")]
        mesh = None

        for c, (ds, name) in enumerate(panels):
            ax = axes[r, c]
            field = SelectHour(ds, var, time)

            # --- blank panel if no data for this hour ---
            if field is None:
                ax.set_axis_off()
                ax.text(0.5, 0.5, "No data", transform=ax.transAxes,
                        ha="center", va="center", fontsize=fontSettings["labelFont"])
                ax.set_title(name, fontsize=fontSettings["labelFont"], fontweight="bold")
                continue

            # --- mask no-echo / no-rain ---
            if var == "refl1km_hourly_max":
                field = field.where(field > reflMin)      # refl <= 0 dBZ -> NaN
            else:
                field = field.where(field >= rainMin)     # rain < rainMin -> NaN

            if var == "refl1km_hourly_max":
                # --- reflectivity: PlotReflectivity adds colormap, features, gridlines ---
                mesh = RadarPlotting_Class.PlotReflectivity(
                    ax, field.latitude, field.longitude, field,
                    dataName=name, timeTitle="", showLat=(c == 0),
                    tickFont=fontSettings["tickFont"])
            else:
                # --- rain rate: same map features as PlotReflectivity (gray land) ---
                mesh = ax.pcolormesh(field.longitude, field.latitude, field,
                                     vmin=vmin, vmax=vmax, cmap=cmap,
                                     transform=ccrs.PlateCarree(), zorder=20)
                ax.add_feature(RadarPlotting_Class.COAST, linewidth=1)
                ax.add_feature(RadarPlotting_Class.BORDERS, linewidth=0.8)
                ax.add_feature(RadarPlotting_Class.STATES, linewidth=0.5)
                ax.add_feature(RadarPlotting_Class.LAND, facecolor="lightgray", alpha=0.3)
                ax.add_feature(RadarPlotting_Class.LAKES, edgecolor="k", facecolor="none")
                gl = ax.gridlines(draw_labels=True, color="gray", alpha=0.5,
                                  linestyle="--", zorder=21)
                gl.top_labels = gl.right_labels = False
                gl.left_labels = (c == 0)
                gl.xlabel_style = gl.ylabel_style = {"size": fontSettings["tickFont"]}
                
            ax.set_title(name, fontsize=fontSettings["labelFont"], fontweight="bold")

        # --- one colorbar per row, shared by all three panels ---
        if mesh is not None:
            cbar = fig.colorbar(mesh, ax=axes[r, :], orientation="vertical",
                                shrink=1.0, aspect=15, pad=0.01)
            if var == "refl1km_hourly_max":
                # same ticks/format as the reflectivity comparison figures
                _, _, _, ticks = RadarPlotting_Class.GetReflectivityColormap()
                RadarPlotting_Class.FormatReflectivityColorbar(cbar, ticks, "vertical", False)
            cbar.set_label(label, fontsize=fontSettings["labelFont"])
            cbar.ax.tick_params(labelsize=fontSettings["tickFont"])


    # --- main title above the top-middle panel (larger, bold) ---
    # timeStr = pd.to_datetime(time).strftime("%m-%d %H")
    axes[0, 1].text(0.5, 1.08,
                    f"{ds_NSSL.attrs['region']} {ds_NSSL.attrs['case']}",
                    transform=axes[0, 1].transAxes, ha="center", va="bottom",
                    fontsize=fontSettings["titleFont"], fontweight="bold")
    return fig

def SelectHour(ds, var, time):
    # Return ds[var] at this hour, or None if the dataset/variable/hour is missing
    if ds is None or var not in ds:
        return None
    try:
        return ds[var].sel(time=time)
    except KeyError:
        return None

timeStringDictionary = {
    "TRACER": {
        "WET":     {0:  "2022-07-01_18.00.00"},
        "DIURNAL": {-5: "2022-06-22_21.00.00"},
    },
    "PRECIP": {
        "WET":     {12: "2022-06-07_03.00.00"},
        "DIURNAL": {12: "2022-07-16_05.00.00"},
    },
    "Hawaii": {
        "WET":    {12: "2021-12-06_23.00.00"},
        "TRADES": {24: "2022-08-08_00.45.00"},   # or "2022-08-08_01.00.00"
    },
}

def GetPlotTimeUTC(ModelData, timeStringDictionary):
    # Look up the snapshot time for this region/case/spinup and convert it to the
    # "YYYY-MM-DDTHH:MM" string that PlotHourlyMax expects.
    ts = timeStringDictionary[ModelData.region][ModelData.case][int(ModelData.spinup_hours)]
    timeUTC = datetime.strptime(ts, "%Y-%m-%d_%H.%M.%S").strftime("%Y-%m-%dT%H:%M")
    print(timeUTC)
    return [timeUTC]

################################################################################################

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300
fontSettings = {
    "tickFont": 15,
    "labelFont": 15,
    "legendFont": 14,
    "titleFont": 22,
}

In [ ]:
def PlotStratiformFractionTimeseries(series, unitsLabel="%", skipHours=12):
    """
    2x2 time series: rows = reflectivity, rain; columns = stratiform fraction of
    precipitation, stratiform fraction of domain. Lines: NSSL, observations, TEMPO.
    Observation names come from STRAT_DISPLAY_NAMES (e.g., MRMS / S-Pol).
    """
    region = ModelData_NSSL.region; case = ModelData_NSSL.case

    # --- drop the first skipHours hours ---
    t = series.index.get_level_values("time")
    series = series[t >= t.min() + pd.Timedelta(hours=skipHours)]

    colors = {"NSSL": "blue", "MRMS": "black", "TEMPO": "green"}
    rows = [("refl", "Reflectivity"), ("rain", "Rain-rate")]
    cols = [("strat_frac_of_precip", "Stratiform / (Stratiform + Convective)"),
            ("strat_frac_of_domain", "Stratiform / Domain")]

    fig, axes = plt.subplots(2, 2, figsize=(16, 9), sharex=True,
                             gridspec_kw={"wspace": 0.1, "hspace": 0.15})

    for r, (field, rowTitle) in enumerate(rows):
        for c, (metric, colTitle) in enumerate(cols):
            ax = axes[r, c]
            for scheme, color in colors.items():
                label = f"{scheme} {field}"
                # --- one line per scheme/obs; skip if that dataset was not provided ---
                if label not in series.index.get_level_values("label"):
                    continue
                s = series.xs(label, level="label")[metric]
                ax.plot(s.index, s.values, color=color, marker="o", ms=3,
                        label=STRAT_DISPLAY_NAMES[field][scheme])

            ax.set_title(f"{rowTitle}: {colTitle}", fontsize=fontSettings["labelFont"] + 2)
            if c == 0:
                ax.set_ylabel(f"{unitsLabel}", fontsize=fontSettings["labelFont"])
            ax.grid(True)
            ax.tick_params(axis="both", labelsize=fontSettings["tickFont"])
            if r == 1:
                ax.set_xlabel("Time (UTC)", fontsize=fontSettings["labelFont"])
                plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
            # if c == 0:
            #     ax.legend(fontsize=fontSettings["legendFont"])   # both rows: obs names differ

    fig.suptitle(f"{region} {case}", fontsize=fontSettings["titleFont"], fontweight="bold")
    fig.tight_layout()
    return [fig,series]

STRAT_DISPLAY_NAMES = {
    "refl": {"NSSL": "NSSL", "MRMS": "MRMS / S-Pol", "TEMPO": "TEMPO"},
    "rain": {"NSSL": "NSSL", "MRMS": "MRMS QPE / QPESUMS", "TEMPO": "TEMPO"},
}

def GetMeanStratiformTableLines(series, title=f"{ModelData_NSSL.region} {ModelData_NSSL.case}", unitsLabel="%"):
    """Builds the mean-stratiform table as a list of text lines."""
    metrics = [("strat_frac_of_precip", "Strat / (Strat + Conv)"),
               ("strat_frac_of_domain", "Strat / Domain")]
    fields = [("refl", "Reflectivity"), ("rain", "Rain-rate")]
    schemes = ["NSSL", "MRMS", "TEMPO"]
    labels = series.index.get_level_values("label")

    # --- layout: name column, then "mean (n hours)" per metric ---
    w0, w = 24, 24
    line = "+" + "-" * (w0 + 2) + ("+" + "-" * (w + 2)) * len(metrics) + "+"
    full = w0 + (w + 3) * len(metrics) + 1
    out = [line]
    if title:
        out += ["| " + f"{title}  (mean {unitsLabel} over plotted hours)".ljust(full) + "|", line]
    out += ["| " + "".ljust(w0) + " | " + " | ".join(m[1].center(w) for m in metrics) + " |", line]

    for field, fieldTitle in fields:
        out.append("| " + fieldTitle.ljust(full) + "|")
        for scheme in schemes:
            label = f"{scheme} {field}"
            if label not in labels:
                continue                      # dataset not provided
            s = series.xs(label, level="label")
            cells = []
            for metric, _ in metrics:
                vals = s[metric].dropna()     # NaN hours are skipped
                cells.append(f"{vals.mean():6.1f} {unitsLabel}  (n={len(vals)})".center(w)
                             if len(vals) else "--".center(w))
            name = "  " + STRAT_DISPLAY_NAMES[field][scheme]
            out.append("| " + name.ljust(w0) + " | " + " | ".join(cells) + " |")
        out.append(line)
    return out


def PrintMeanStratiformTable(series, title=f"{ModelData_NSSL.region} {ModelData_NSSL.case}", unitsLabel="%"):
    print("\n".join(GetMeanStratiformTableLines(series, title, unitsLabel)))

def SaveMeanStratiformTable(series, ModelData, title=f"{ModelData_NSSL.region} {ModelData_NSSL.case}", unitsLabel="%", dpi=300):
    """Saves the table as .txt and as a monospace .png snapshot next to the figures."""
    lines = GetMeanStratiformTableLines(series, title, unitsLabel)

    # --- same folder and naming pattern as SaveFigure ---
    saveDirectory = os.path.join(outputPlottingDirectory,
                                 f"{ModelData.region}_{ModelData.case}_{ModelData.spinup_hours}hrs")
    os.makedirs(saveDirectory, exist_ok=True)
    fileStem = f"{dataType}_StratiformTable"

    # --- text file ---
    txtPath = os.path.join(saveDirectory, fileStem + ".txt")
    with open(txtPath, "w") as f:
        f.write("\n".join(lines) + "\n")

    # --- image snapshot: render the lines in a monospace font ---
    fig = plt.figure(figsize=(0.105 * len(lines[0]), 0.26 * len(lines) + 0.3))
    fig.text(0.01, 0.99, "\n".join(lines), family="monospace", fontsize=10, va="top", ha="left")
    pngPath = os.path.join(saveDirectory, fileStem + ".png")
    fig.savefig(pngPath, dpi=dpi, bbox_inches="tight", pad_inches=0.05)
    plt.close(fig)
    print(f"Saved table: {txtPath}\nSaved table: {pngPath}")

In [ ]:
def SaveFigure(fig, ModelData1, ModelData2, dpi=300,
               plotType="StratiformFraction"):
    """
    Saves a figure to the appropriate directory based on the models in combinedDict.
    """
    # --- Define output subdirectory and file path ---
    outputSubDirectory = f"{ModelData1.region}_{ModelData1.case}_{ModelData1.spinup_hours}hrs"
    os.makedirs(os.path.join(outputPlottingDirectory, outputSubDirectory), exist_ok=True)

    outputFilePath = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory,
        f"{plotType}.jpg"
    )

    # # --- Save figure ---
    fig.savefig(outputFilePath, dpi=dpi, bbox_inches="tight",pad_inches=0.02)
    # plt.close(fig)
    print(f"Saved image: {outputFilePath}")

In [ ]:
###################
#CALCULATIONS

In [ ]:
#Model Calculations
hourlyMax_NSSL = LoadOrRun_GetHourlyMax_Model(ModelData_NSSL)
hourlyMax_TEMPO = LoadOrRun_GetHourlyMax_Model(ModelData_TEMPO)

In [ ]:
# Observation Calculation
hourlyMax_obsRain = GetObservations_Precipitation(ModelData_NSSL)
hourlyMax_obsRefl = GetObservations_Reflectivity(ModelData_NSSL, DirectoryManager)

In [ ]:
#Calculating Stratiform Fraction Time-series
stratiformFractionSeries = LoadOrRun_GetStratiformTimeseries(ModelData_NSSL,hourlyMax_NSSL,hourlyMax_TEMPO,
                                                ds_obsRefl=hourlyMax_obsRefl,ds_obsRain=hourlyMax_obsRain)

In [ ]:
###################
#PLOTTING

In [ ]:
#Single Time Plots
[timeUTC] = GetPlotTimeUTC(ModelData_NSSL, timeStringDictionary)
fig = PlotHourlyMax(hourlyMax_NSSL, hourlyMax_TEMPO,
                    ds_obsRain=hourlyMax_obsRain, ds_obsRefl=hourlyMax_obsRefl,
                    timeUTC=timeUTC)

In [ ]:
#Stratiform Fraction Time-series
[fig,series] = PlotStratiformFractionTimeseries(stratiformFractionSeries)
SaveFigure(fig, ModelData_NSSL, ModelData_TEMPO, plotType="StratiformFraction")

# --- print and save the mean of each plotted time series ---
PrintMeanStratiformTable(series)
SaveMeanStratiformTable(series, ModelData_NSSL)